# AIML ZG519 – Natural Language Processing Applications
# Assignment: Sentiment Analysis – A Comparative Study

**Session Reference:** Session 6 (Sentiment Analysis) — builds on Sessions 3–5 (RAG, QA, Conversational AI)

**Group No.:** `<Your Group Number>`
**Problem Statement:** `<PS1-PS7, see Assignment Brief Section 3>`
**Group Members (Name, BITS ID):**
- `<Member 1>`
- `<Member 2>`
- `<Member 3>` `(add/remove rows as needed)`

---

## Objective
In this assignment you will build **four different sentiment analysis pipelines** on the same
dataset — classical Machine Learning, Deep Learning, a fine-tuned Transformer, and a
Generative AI / LLM prompting approach — and compare them on accuracy, robustness,
interpretability, latency and cost.

**Total Marks: 10**

Fill in every cell marked `# TODO`. Do not delete the markdown instructions — they are part
of your submission and will be used for evaluation. Answer the **reflection questions** at the
end of each part in the markdown cell provided.

## Learning Outcomes
- LO2: Hands-on experience with major NLP technologies and tools
- LO3: Apply NLP techniques (sentiment analysis) using state-of-the-art Gen AI techniques
- LO4: Sharpen programming skills for NLP applications


## 0. Environment Setup

Run this cell first. If you are on Google Colab, uncomment the `pip install` line.


In [ ]:
# !pip install -q scikit-learn nltk gensim tensorflow transformers datasets torch openai matplotlib seaborn pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re, time, json, random

random.seed(42)
np.random.seed(42)

sns.set_style("whitegrid")
%matplotlib inline


## 1. Dataset — Your Assigned Problem Statement

Your group has been assigned ONE problem statement (PS1–PS7) in the **Assignment Brief**,
based on your group number. Fill in the table below with your group's details before doing
anything else — the TA will cross-check this against the group allocation list.

| Field | Your Group's Value |
|---|---|
| Group Number | `<TODO>` |
| Problem Statement ID | `<TODO — e.g. PS3>` |
| Domain | `<TODO — e.g. Aviation / Travel>` |
| Dataset used | `<TODO — exact dataset name + source>` |
| Aspect focus (for Parts F & G) | `<TODO — copy from the brief's PS table>` |

| PS | Domain | Suggested Dataset | Aspect Focus |
|---|---|---|---|
| PS1 | E-Commerce Product Reviews | `amazon_polarity` (one category) or Flipkart Reviews (Kaggle) | price, delivery, quality, packaging |
| PS2 | Movie / OTT Reviews | `imdb` or Rotten Tomatoes | acting, plot, cinematography, pacing |
| PS3 | Airline & Travel | Twitter US Airline Sentiment (Kaggle) | delay, crew behaviour, baggage, comfort |
| PS4 | Restaurant / Food-Delivery | `yelp_polarity` or Zomato Reviews (Kaggle) | taste, delivery time, hygiene, packaging |
| PS5 | Healthcare: Patient Drug Reviews | UCI Drug Review Dataset (Kaggle) | effectiveness, side effects, ease of use |
| PS6 | Financial News & Market Sentiment | `financial_phrasebank` or Twitter Financial News Sentiment | earnings, macro risk, bullish/bearish tone |
| PS7 | Mobile App Store Reviews | Google Play Store App Reviews (Kaggle) or `app_reviews` (HuggingFace) | app crashes/bugs, UI/UX, feature requests, pricing |

> **Tip:** Sample 5,000–10,000 rows (stratified) if your dataset is larger, so classical ML and
> deep learning training stay fast in Colab. Keep the same sample for ALL four approaches
> (Parts B–E) so comparisons are fair. If your assigned dataset naturally has fewer rows
> (e.g. PS3, PS6), use the full dataset and note the smaller size in your report.
>
> **PS5 groups:** do not frame any output as medical advice or diagnosis — see the brief's note
> tying this to Session 7 (Privacy & Ethics in NLP).
>
> **PS7 groups:** star ratings (1–5) can be mapped to sentiment labels (e.g. 1–2 star =
> Negative, 3 = Neutral, 4–5 = Positive) if your chosen dataset doesn't ship explicit labels.


In [ ]:
# TODO: Load the dataset specified for YOUR assigned problem statement here.
# Example using HuggingFace `datasets` for PS2 (IMDB):
#
# from datasets import load_dataset
# raw = load_dataset("imdb")
# train_df = pd.DataFrame(raw["train"]).sample(5000, random_state=42).reset_index(drop=True)
# test_df  = pd.DataFrame(raw["test"]).sample(1500, random_state=42).reset_index(drop=True)
#
# For Kaggle-hosted datasets (PS1/PS3/PS4/PS5/PS7 options), download via the Kaggle API or
# manual upload, then pd.read_csv(...) as usual.

train_df = None   # TODO
test_df  = None   # TODO

# TODO: Print shape, class balance, and a few sample rows


### 1.1 Exploratory Data Analysis (EDA)

Produce at minimum:
- Class distribution plot
- Review-length distribution (word count histogram)
- Word clouds or top-N frequent tokens per class

**Reflection Q1:** Is the dataset balanced? How might class imbalance affect model choice
and evaluation metric selection later in this notebook?


In [ ]:
# TODO: EDA — class balance
# TODO: EDA — text length distribution
# TODO: EDA — top frequent words per class (after basic cleaning)


_Your answer to Reflection Q1:_ `TODO`

## 2. Part A — Text Preprocessing (1 mark)

Implement a reusable `clean_text()` function. Keep two versions of the text:
- `text_classical`: heavily cleaned (lowercase, punctuation/stopword removal, lemmatization) —
  for the classical ML pipeline (Part 3).
- `text_neural`: lightly cleaned (lowercase, HTML/URL removal only) — for DL/Transformer/LLM
  pipelines (Parts 4–5), since those models learn their own representations and over-cleaning
  can hurt them.


In [ ]:
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text_light(text: str) -> str:
    """Minimal cleaning for DL/Transformer/LLM pipelines."""
    text = re.sub(r"<.*?>", " ", text)          # strip HTML
    text = re.sub(r"http\S+", " ", text)         # strip URLs
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

def clean_text_classical(text: str) -> str:
    """Heavier cleaning for TF-IDF / BoW pipelines."""
    text = clean_text_light(text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [lemmatizer.lemmatize(t) for t in text.split() if t not in stop_words and len(t) > 2]
    return " ".join(tokens)

# TODO: apply both functions to train_df / test_df and store as new columns
# train_df['text_classical'] = train_df['text'].apply(clean_text_classical)
# train_df['text_neural']    = train_df['text'].apply(clean_text_light)


## 3. Part B — Classical Machine Learning (2 marks)

Build **two** classical pipelines using `TfidfVectorizer` (or `CountVectorizer`) with:
1. **Naive Bayes** (`MultinomialNB`)
2. **One of:** Logistic Regression / Linear SVM

For each: report Accuracy, Precision, Recall, F1 (weighted), and a confusion matrix.
Tune at least one hyperparameter (e.g. `ngram_range`, `max_features`, `C`) using
`GridSearchCV` or `RandomizedSearchCV` and report the best configuration.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# TODO: Vectorize text_classical with TfidfVectorizer

# TODO: Train MultinomialNB, evaluate on test set

# TODO: Train LogisticRegression or LinearSVC, evaluate on test set

# TODO: Hyperparameter tuning with GridSearchCV on ONE model; report best params + score


In [ ]:
# TODO: Plot confusion matrices for both classical models side-by-side (use plt.subplots)


**Reflection Q2:** Which classical model performed better and why do you think that is,
given the nature of TF-IDF features? What are the main limitations of a bag-of-words
representation for sentiment analysis (e.g. negation handling, word order, sarcasm)?

_Your answer:_ `TODO`


## 4. Part C — Deep Learning (2 marks)

Build a neural sentiment classifier using an embedding layer + a recurrent or convolutional
architecture (choose ONE): `BiLSTM`, `LSTM`, `GRU`, or `1D-CNN` over word embeddings (Keras/
TensorFlow or PyTorch). Use **pre-trained embeddings** (e.g. GloVe) if possible, otherwise
train an embedding layer from scratch and note the trade-off.

Report: training/validation loss & accuracy curves, test accuracy, precision, recall, F1,
and confusion matrix. Compare training time and number of parameters against the classical
model.


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, GlobalMaxPooling1D, Conv1D

MAX_VOCAB = 20000
MAX_LEN = 200

# TODO: Tokenize text_neural, pad sequences to MAX_LEN

# TODO: Build a Sequential model: Embedding -> Bidirectional(LSTM) -> Dense -> Dense(sigmoid/softmax)

# TODO: Compile (optimizer='adam', loss=..., metrics=['accuracy']) and model.summary()

# TODO: Train with EarlyStopping callback, keep the History object for plotting


In [ ]:
# TODO: Plot training vs validation accuracy/loss curves (2 subplots)

# TODO: Evaluate on test set -> accuracy, F1, confusion matrix

# TODO: Print total trainable parameters and total training time (use time.time())


**Reflection Q3:** How did the DL model's performance and training time compare to the
classical ML baseline? Did pre-trained embeddings help? What would you try next to improve
this model (e.g. attention, more data, transfer learning)?

_Your answer:_ `TODO`


## 5. Part D — Transformer Fine-Tuning (2 marks)

Fine-tune a pre-trained transformer (`distilbert-base-uncased` recommended for speed, or
`bert-base-uncased`) on your sentiment dataset using the HuggingFace `transformers` +
`datasets` libraries (`Trainer` API, or a manual PyTorch training loop).

If compute is limited, fine-tune on a **smaller subset** (e.g. 2,000 train / 500 test) and
state this clearly — the comparison is still valid as long as it is fair (i.e. compare all
approaches on the same subset where possible, or clearly annotate the sample-size difference).


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

MODEL_NAME = "distilbert-base-uncased"

# TODO: Load tokenizer and model (num_labels = number of classes in your dataset)

# TODO: Tokenize train/test sets (truncation=True, padding=True, max_length=256)

# TODO: Wrap into a torch Dataset / HF Dataset object

# TODO: Define TrainingArguments (small num_train_epochs=2-3 for time budget) and Trainer

# TODO: trainer.train()


In [ ]:
# TODO: trainer.evaluate() -> accuracy, F1, precision, recall

# TODO: Confusion matrix on test predictions

# TODO: Record fine-tuning time and inference latency per 100 samples (time.time())


**Reflection Q4:** How does the fine-tuned transformer compare to the DL model from
Part B on accuracy vs. training time vs. compute cost? In what production scenario would
you NOT choose a transformer despite better accuracy?

_Your answer:_ `TODO`


## 6. Part E — Generative AI / LLM Prompting Approach (1.5 marks)

Instead of training a model, use an **LLM via prompting** to perform sentiment classification.
Implement and compare:
1. **Zero-shot prompting** — ask the LLM to classify sentiment with no examples.
2. **Few-shot prompting** — include 3–5 labelled examples in the prompt.

You may use the OpenAI API, Anthropic API, or any open-source LLM you can run locally
(e.g. via `transformers` `pipeline("sentiment-analysis")` is NOT sufficient here — that is a
fine-tuned classifier, not prompting). Evaluate on a **sample** of your test set (e.g. 100–200
rows) to control API cost.


In [ ]:
# --- Example scaffold using the OpenAI-compatible chat completion style. ---
# --- Replace with your provider of choice (OpenAI / Anthropic / local LLM). ---
# import openai
# openai.api_key = "YOUR_API_KEY"

def zero_shot_prompt(review_text: str) -> str:
    return f"""Classify the sentiment of the following review as exactly one word:
Positive, Negative, or Neutral.

Review: \"{review_text}\"

Sentiment:"""

def few_shot_prompt(review_text: str, examples: list) -> str:
    # TODO: build a prompt that prepends 3-5 (review, label) example pairs, then asks
    # for the label of `review_text`
    pass

def call_llm(prompt: str) -> str:
    # TODO: call your chosen LLM API/model and return the raw text response
    # response = openai.chat.completions.create(model="gpt-4o-mini",
    #                messages=[{"role": "user", "content": prompt}], temperature=0)
    # return response.choices[0].message.content.strip()
    raise NotImplementedError

# TODO: sample ~150 rows from test_df
# TODO: run zero_shot_prompt over the sample, parse predicted label, measure latency + (est.) cost
# TODO: run few_shot_prompt over the same sample
# TODO: compute accuracy / F1 for each prompting strategy, compare to Parts 3-5


**Reflection Q5:** Did few-shot prompting improve over zero-shot? How did LLM prompting
accuracy compare to the fine-tuned transformer — and how do the *costs* compare (API cost
per 1,000 predictions vs. one-time fine-tuning + free inference)? When would you prefer
prompting over fine-tuning in a real product?

_Your answer:_ `TODO`


## 7. Part F — Comparative Analysis & Report (1.5 marks)

Consolidate results from Parts 3–6 into a single comparison table and at least one chart.
Discuss trade-offs beyond raw accuracy: **training/setup time, inference latency, cost,
data requirements, interpretability, and robustness** (e.g. test each model on 5 tricky
examples you write yourself — negation, sarcasm, mixed sentiment — and report how each
approach handles them).


In [ ]:
results = pd.DataFrame({
    "Approach": ["Naive Bayes", "LogReg/SVM", "BiLSTM/CNN", "Fine-tuned Transformer",
                 "LLM Zero-shot", "LLM Few-shot"],
    "Accuracy": [None]*6,     # TODO fill from your runs above
    "F1 (weighted)": [None]*6,
    "Train/Setup Time (s)": [None]*6,
    "Inference Latency (ms/sample)": [None]*6,
    "Approx. Cost": ["Free", "Free", "Free (compute)", "Free (compute, one-time)",
                      "$ per call", "$ per call (higher, longer prompt)"],
})
results


In [ ]:
# TODO: Bar chart comparing Accuracy and F1 across all approaches
# TODO: (Optional) Bar/line chart comparing latency across approaches (log scale often helps)


> Write your own 5 stress-test examples **in your assigned domain** (see the Aspect Focus
> column for your PS in the brief), covering: negation, sarcasm, mixed sentiment, a
> neutral/ambiguous case, and a double negation. The placeholders below are illustrative only
> — replace them with domain-appropriate sentences before running.


In [ ]:
# --- Robustness stress-test: REPLACE with 5 examples from YOUR assigned domain ---
tricky_examples = [
    "TODO: negation example in your domain",
    "TODO: sarcasm example in your domain",
    "TODO: mixed-sentiment example in your domain",
    "TODO: neutral/ambiguous example in your domain",
    "TODO: double-negation example in your domain",
]

# TODO: run each of your trained/prompted approaches on `tricky_examples` and tabulate
# predictions side-by-side to compare qualitatively


### Final Written Summary (Required)

Write a 250–400 word summary covering:
1. Which approach would you deploy in production for a **high-traffic, cost-sensitive**
   consumer app (e.g. e-commerce review moderation), and why?
2. Which approach would you deploy for a **low-volume, high-stakes** use case (e.g.
   analyzing patient feedback in healthcare), and why?
3. One ethical/privacy consideration relevant to sending user-generated text (reviews,
   messages) to a third-party LLM API, referencing the course's Session 7 (Privacy & Ethics
   in NLP).

_Your summary:_ `TODO`


## Submission Checklist

- [ ] All `# TODO` code cells completed and executed top-to-bottom without errors
- [ ] All 5 reflection questions answered
- [ ] Comparison table (Part F) fully populated with real numbers
- [ ] Final written summary completed
- [ ] Notebook renamed to `<PSid>_Group<GroupNo>_NLP_SentimentAnalysis_Assignment.ipynb` (e.g. `PS3_Group12_NLP_SentimentAnalysis_Assignment.ipynb`)
- [ ] Group number, problem statement, and all group members' names/IDs filled in at the top
- [ ] (If using paid LLM APIs) API keys removed before submission — do NOT commit secrets

**Academic Integrity:** You may use AI coding assistants to help write boilerplate code, but
the analysis, reflections, and final summary must be your own original interpretation of
your results.
